<a href="https://colab.research.google.com/github/khainguyen1108/Add2BigNumber/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Tải và cài đặt Unsloth bản cập nhật mới nhất
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Cài đặt các thư viện phụ trợ (để nguyên không ép version xformers)
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7gpqyi93/unsloth_8e10e71c511f4bd8ad9f0c7f015475bd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7gpqyi93/unsloth_8e10e71c511f4bd8ad9f0c7f015475bd
  Resolved https://github.com/unslothai/unsloth.git to commit 93a24f66980b2820083d2882fd3a720d894a1414
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.2 MB/s eta 0:00:00


In [2]:
from unsloth import FastLanguageModel
import torch

# Kích thước bộ nhớ ngữ cảnh (độ dài tối đa của câu hỏi + câu trả lời)
max_seq_length = 2048
dtype = None
load_in_4bit = True # Ép model dùng bản nén 4-bit để vừa với RAM của Colab T4

# 1. Kéo Model gốc từ Hugging Face về Server Colab
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Gắn LoRA Adapter (Chỉ định các vùng não bộ sẽ được update)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Độ lớn của Adapter (thường dùng 8, 16, 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Load Model gốc & gắn LoRA thành công!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Load Model gốc & gắn LoRA thành công!


In [3]:
from datasets import Dataset

# 1. Định nghĩa cái "Khuôn" (Template) để dạy AI
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Nút 'Stop' cực kỳ quan trọng để AI biết lúc nào nói xong thì ngậm miệng lại

# 2. Hàm để ép dữ liệu của bạn vào cái khuôn ở trên
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Ghép 3 thành phần lại và gắn thêm EOS_TOKEN ở cuối
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 3. Tạo một bộ dữ liệu nhỏ để test (Sau này bạn sẽ thay bằng file của bạn)
my_data = {
    "instruction": [
        "Viết một script bằng Golang để tự động upload video lên YouTube.",
        "Giải thích cho tôi cách cấu hình Nginx để chặn các cuộc tấn công Slowloris."
    ],
    "input": [
        "Sử dụng YouTube Data API v3 và OAuth2.",
        "Server đang chạy Ubuntu."
    ],
    "output": [
        "Đây là đoạn code Golang mẫu để upload video sử dụng thư viện `google.golang.org/api/youtube/v3`...\n\n(Code mẫu giả định...)",
        "Để chống Slowloris trên Nginx, bạn cần giới hạn kết nối và timeout. Mở file `nginx.conf` và thêm cấu hình: `client_body_timeout 10s; client_header_timeout 10s;`..."
    ]
}

# Chuyển đổi dữ liệu và nạp vào Template
dataset = Dataset.from_dict(my_data)
dataset = dataset.map(formatting_prompts_func, batched = True,)

print("Nạp dữ liệu vào Template thành công! Dataset có", len(dataset), "câu hỏi.")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Nạp dữ liệu vào Template thành công! Dataset có 2 câu hỏi.


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Chạy thử 60 bước cho nhanh. Khi train thật có thể đổi thành num_train_epochs = 3
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1, # Cứ 1 bước in ra log 1 lần
        optim = "adamw_8bit", # Trình tối ưu hóa siêu tiết kiệm RAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Bắt đầu huấn luyện...")
trainer_stats = trainer.train()
print("Huấn luyện xong!")

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

Bắt đầu huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.729382
2,2.729382
3,2.653194
4,2.408830
5,2.067487
6,1.658380
7,1.203363
8,0.762924
9,0.429648
10,0.214603


Huấn luyện xong!


In [5]:
# 1. Bật chế độ suy luận nhanh của Unsloth (Giúp tốc độ sinh chữ tăng gấp 2 lần)
FastLanguageModel.for_inference(model)

# 2. Chuẩn bị câu hỏi (Đưa vào khuôn Alpaca)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Viết cho tôi một hàm Golang để kết nối với cơ sở dữ liệu MySQL.", # Yêu cầu (Instruction)
        "Sử dụng thư viện GORM.", # Ngữ cảnh (Input) - Có thể để trống "" nếu không có
        "", # Phần Response phải để trống để AI điền vào
    )
], return_tensors = "pt").to("cuda")

print("Đang suy nghĩ...\n")

# 3. Ép AI sinh chữ (max_new_tokens là số lượng chữ tối đa nó được phép nói)
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)

# 4. Dịch các con số trả về thành văn bản đọc được
cau_tra_loi = tokenizer.batch_decode(outputs)[0]
print(cau_tra_loi)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đang suy nghĩ...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Viết cho tôi một hàm Golang để kết nối với cơ sở dữ liệu MySQL.

### Input:
Sử dụng thư viện GORM.

### Response:
Đây là đoạn code Golang mẫu để tự động cấu hình và kết nối với một server MySQL:...

(Code mẫu giả định...)<|end_of_text|>
